# 第二章：运动学——从位姿到约束运动

运动学（Kinematics）构成了机器人操纵（Manipulation）的几何基础。然而，在现代机器人系统中，运动学的功能远不止于建立关节角度和末端执行器姿态之间的数学映射。在“任务与运动规划”（Task and Motion Planning, TAMP）的框架下，运动学（尤其是逆运动学, Inverse Kinematics, IK）扮演着更为复杂且关键的角色：它是连接高层符号逻辑与低层几何现实的桥梁。


## I. 运动学在任务与运动规划（TAMP）中的核心作用


### A. 超越位姿求解：运动学作为TAMP的瓶颈

TAMP 框架的核心是将规划分为两个层面：高层的符号任务规划（Symbolic Task Planner）和底层的几何运动规划（Motion Planner）¹。TAMP 系统的目标是自主解决由多个子任务组成的“长时程”（long-horizon）操纵问题¹。

在这种分层结构中，符号规划器生成一系列抽象的动作，例如 `$pick(obj1)$` 或 `$place(obj1, region2)$`。而运动规划器（及其核心组件IK）则负责检查这些抽象动作在物理世界中是否可行（geometrically feasible）²。

然而，运动规划（包括IK可行性检查）是一个计算密集型过程¹。TAMP 系统必须进行大量的几何查询，以找到一个在符号层和几何层都有效的规划。这种高计算开销是TAMP的核心瓶颈，常常导致其难以应用于需要快速反应的真实世界场景¹。

<figure style="text-align: center; margin: 20px 0;">
    <video controls style="display: block; margin: 0 auto; max-width: 90%; height: auto;" autoplay muted loop>
        <source src="https://kyamagu.github.io/teaching/tamp/2019/videos/video-tamp.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            一个典型的TAMP问题：布置餐桌。机器人需要自主推理出动作顺序（任务规划）并为每一步找到可行的轨迹（运动规划）。
    </figcaption>
</figure>


### B. 几何可行性：为符号动作提供几何“锚定”（Grounding）

在TAMP中，IK的主要作用是“推理”（reason about）符号规划器所生成的动作序列中，关键位姿（如抓取前、抓取时、放置时）的可行性²。一个至关重要的策略是，TAMP系统通常不会立即生成完整的、无碰撞的轨迹。相反，它们首先采用一种“乐观”的规划策略³：仅仅使用IK来检查这些关键的初始和最终位姿是否可达²。

如果这些关键帧（keyframes）是运动学可达的，规划器就乐观地假设它们之间的路径可能存在，并将这个动作作为一个可行的步骤纳入高层规划。这种“先检查后规划”的模式，为抽象的符号动作提供了具体的几何“锚定”（Grounding）¹。


### C. IK“流”（Stream）：规划器的可行性“神谕”（Oracle）

在形式化的TAMP系统（如 PDDLStream）中，IK求解器被抽象为一个计算“流”（Stream）或“神谕”（Oracle）³。任务规划器不再处理具体的关节角度，而是生成一个对IK流的查询。例如，一个典型的查询可能如下所示⁴：

```lisp
(:stream inverse-kinematics
 :inputs (?r ?o ?p ?g)
 :domain (and (Robot ?r) (Pose ?o ?p) (Grasp ?o ?g))
 :outputs (?q ?t))
```

这个查询的语义是：“对于机器人`?r`、物体`?o`、物体位姿`?p` 和抓取`?g`，是否存在一个可行的关节配置`?q`（以及可选的轨迹`?t`）？” TAMP规划器会“乐观地”假设这个流会返回一个有效的输出（一个可行的 `$q$`）³，并且只在绝对必要时才调用这个计算成本高昂的流³。


### D. TAMP为何使IK“更难”：解的选择问题

TAMP规划器对IK的“流”抽象⁴与其计算瓶颈的现实¹之间存在着根本性的矛盾。规划器希望IK是一个简单的、快速返回布尔值（是/否）的函数。但现实是，IK是一个复杂的、可能很慢的、并且（对于冗余机器人）返回无限解集的问题。

这种对快速可行性检查的迫切需求¹，催生了不求解IK，而只是预测其可行性的方法。例如，引入神经可行性分类器（Neural Feasibility Classifier, NFC）作为一种视觉启发式方法（visual heuristic）¹。NFC 通过分析机器人工作空间的图像，利用卷积神经网络（CNN）来分类一个提议的动作是否可行，从而避免了昂贵的IK求解和运动规划¹。

这引出了TAMP中IK的核心挑战：TAMP规划器不能只问“是否存在解？”，而必须问一个更难的问题：“**是否存在一个好的解？**”——即，一个不仅运动学可达，而且其邻域（即，从当前配置出发的运动路径）也是可行（例如，无碰撞）的解。


## II. “最佳”IK解：在约束操纵中定义最优性

当IK求解器返回多个（甚至无限个）解时，系统必须选择“最佳”的一个。一个解是否“最佳”或“可行”，是由一系列外在标准定义的⁵。这本质上是一个多目标优化问题⁶。


### A. 冗余问题：选择的“祝福”与“诅咒”

对于一个6D的任务空间位姿（位置+姿态），一个6-DOF（自由度）的机械臂是非冗余的，通常只有有限个解¹¹。而7-DOF或更多自由度的机械臂¹²则具有**运动学冗余**（Kinematic Redundancy）¹²。

冗余性引入了一个“**零空间**”（Null Space）¹²：机械臂可以在保持末端执行器位姿固定不变的情况下，改变其部分关节（例如，“转动手肘”）¹²。

这种冗余性既是“祝福”也是“诅咒”：
- **祝福**：冗余性使机器人能够在满足主要任务（末端位姿）的同时，优化次要目标¹²。例如，主动避开障碍物、避开奇异点、或最大化可操纵性¹²。
- **诅咒**：对于TAMP规划器而言，现在一个IK查询（如 `?q`）返回的不再是一个（或几个）解，而是无限个解（一个连续的流形）¹³。IK“流”必须从这个无限集合中选择一个“最佳”解¹³，这使得IK问题从“求解”转变为“优化”。

<figure style="text-align: center; margin: 20px 0;">
    <img src="https://upload.wikimedia.org/wikipedia/commons/7/77/Null_Space_Motion.gif" 
         width="60%" 
         style="display: block; margin: 0 auto;"> 
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
        零空间运动：7-DOF的KUKA LBR iiwa机器人在末端执行器保持固定的同时，改变其“肘部”姿态。
    </figcaption>
</figure>


### B. 解选择的核心标准（多目标优化）

选择“最佳”解的标准可分为两类：**硬约束**和**软约束**。

#### 1. 硬约束（可行性边界）
这些是必须满足的二元条件，不满足则解无效：
- **关节限制**：解中的所有关节值 $q_i$ 必须在机器人的物理关节范围内 $[\theta_{min}, \theta_{max}]$ ⁵。
- **自碰撞**：解的构型不能导致机器人自身的连杆发生碰撞⁵。
- **环境碰撞**：解的构型不能与工作空间中的已知障碍物发生碰撞⁵。

#### 2. 软约束（优化目标）
当多个解都满足硬约束时，通过这些标量函数 $g(q)$ 来评估解的“质量”：
- **奇异点规避**：解应远离运动学奇异点（Kinematic Singularities）⁵。在奇异点附近，雅可比矩阵（Jacobian matrix）的行列式趋于0¹⁶，微小的笛卡尔运动可能导致关节速度趋于无穷大¹⁶，使得运动不稳定且难以控制⁵。
- **可操纵性**：最大化可操纵性度量（Manipulability）¹⁵，例如雅可比矩阵的条件数或行列式的值¹⁵。这确保机器人在该构型下具有向各个方向运动的良好能力。
- **时间连贯性（平滑度）**：在为轨迹（路径）求解时，这是最重要的标准之一。系统应选择与前一个时间步的解 $q_{t-1}$ 在关节空间中“最接近”的解 $q_t$¹⁴。这最小化了关节运动，避免了不必要的“跳变”。
- **能量/运动最小化**：选择使关节总运动量最小的解¹⁸，或者优先使用“小关节”（如腕部）而非“大关节”（如肩部和肘部）的解¹⁸。


### C. 应用：笛卡尔路径跟踪

笛卡尔路径跟踪（Cartesian path following）是TAMP中一个常见的子问题（例如，焊接、涂胶或擦拭）。它要求末端执行器精确跟踪一条在笛卡尔空间中定义的路径。这暴露了IK解选择的核心挑战。

#### 1. 逐点求解（Naive Solving）的脆弱性

一个常见的“天真”实现是：将笛卡尔路径离散为一系列密集的航点（waypoints），然后独立地为每个航点调用一个数值IK求解器（如LM）⁵。这种方法存在严重缺陷：

- **问题1：奇异点与不连续性**。当路径接近奇异点时，数值解可能会变得不稳定⁵。更糟糕的是，由于数值求解器只保证局部收敛，它可能会在两个（或多个）截然不同的解分支之间“跳跃”（jumps）⁵。例如，从“肘部向上”的解突然跳到“肘部向下”的解。这导致关节空间轨迹的巨大不连续性，这在物理上是不可执行的。
- **问题2：局部最小值陷阱**。数值求解器（如LM）只能保证找到一个局部最优解²⁰。当天真地跟踪路径时，选择的“最近”解（¹⁴）可能会将机器人引导到一个“死胡同”——例如，一个无法继续跟踪路径的奇异构型或关节极限。

<figure style="text-align: center; margin: 20px 0;">
    <img src="https://i.imgur.com/y3yL4q7.gif" 
         width="60%" 
         style="display: block; margin: 0 auto;"> 
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
        当机械臂完全伸展时，它会到达一个奇异点，此时它失去了在特定方向上移动的能力，导致运动不稳定。
    </figcaption>
</figure>

#### 2. 管理不连续性与“重构”（Reconfigurations）

对于某些复杂的笛卡尔路径，不可能存在单一、连续的关节空间轨迹来完成整个任务²¹。这可能是由于关节限制、自碰撞或工作空间中的障碍物。在这种情况下，机器人必须执行一次“**重构**”（Reconfiguration）：机器人暂停任务，偏离参考轨迹，在关节空间中重新定位，然后返回到笛卡尔路径上的同一点，但使用一个不同的IK解（例如，从“肘部向上”切换到“肘部向下”）²¹。重构是非常昂贵的，因为它会增加任务的执行时间和能量消耗，因此必须将其最小化²¹。

#### 3. 案例研究：IKLink - 使用动态规划最小化重构

IKLink (Wang, Sifferman, and Gleicher, 2024) 是一种专门解决此问题的先进图搜索方法²¹。它将笛卡尔路径跟踪问题从一系列独立的“求解”问题，重新构建为一个“规划”问题。其方法论如下²¹：

1.  **生成（Generate）**：IKLink 首先为参考轨迹上的每一个航点生成一个多样的（diverse）IK解集²¹。
2.  **构建图（Build Graph）**：它构建一个分层图（layered graph），其中图的每一层 `i` 对应路径上的第 `i` 个航点，该层的节点就是为该航点找到的IK解集²⁴。
3.  **连接（Link）**：如果机器人可以在时间预算内从层 `i` 的解 $q_A$ 连续（且无碰撞地）移动到层 `i+1$ 的解 $q_B$，则在 $q_A$ 和 $q_B$ 之间添加一条边²⁴。
4.  **优化（Optimize）**：最后，IKLink 使用动态规划（Dynamic Programming）算法²¹ 在这个图中找到一条最优路径。这条路径链接（link）了各个航点的IK解，并保证了最少的重构次数²¹。

IKLink 的方法承认了路径跟踪是一个序列优化问题，而不是一个逐点求解问题。它将IK解的离散集合视为一个图，然后在这个图上搜索最佳路径，从而将TAMP的规划思想引入到低级的运动学控制中。


## III. 逆运动学求解器分类

为了应对TAMP和路径跟踪带来的挑战，研究人员开发了多种IK求解器。它们可以大致分为三类：经典的解析法、经典的数值法和新兴的基于学习的方法。


### A. 经典解析求解器（Analytical Solvers）

解析法（或称封闭解）⁵ 旨在使用代数和几何推导²⁵，为特定的运动学链找到 $q = f^{-1}(T)$ 的精确数学表达式。

- **优点**：计算速度极快（通常在微秒级别），精度高，并且可以返回所有可能的解（通常是有限的，如2、4、8、16个）⁵。
- **缺点**：非通用性（Not general）⁵。公式推导必须针对每种不同的运动学结构独立进行⁵。例如，大多数6R工业机器人满足的Piper准则（拥有球形手腕）可以大大简化求解，但对于一般构型，推导可能极其复杂，甚至需要自动化工具。

#### 自动化解析求解器：ikfast

ikfast（来自OpenRAVE）是一个机器人运动学编译器³¹。它试图解决手动推导的痛苦。

- **方法**：ikfast 不是一个通用的求解器，而是一个**生成器**。它会自动分析机器人的运动学链（通常来自URDF文件³¹），寻找代数上的可解模式，并生成针对该特定机器人的、高度优化的C++求解器代码³¹。
- **优点**：兼具解析解的速度（微秒级）和稳定性³¹。
- **限制**：主要限制在于它不支持高度冗余（>7 DOF）的机械臂³¹，并且在某些复杂情况下可能存在可靠性问题，例如难以处理复杂的关节限制³⁶。


### B. 经典数值求解器（Numerical Solvers）

数值法是通用的⁵。只要我们能计算正向运动学（FK）⁵ 及其雅可比矩阵（FK的一阶导数 $\mathbf{J} = \partial f / \partial q$）³⁷，我们就可以使用迭代法求解任何构型。其核心思想是求解 $e = x_{desired} - f(q)$，通过迭代更新 $q_{k+1} = q_k + \Delta q$ 来使误差 $e$ 最小化。

#### Levenberg-Marquardt (LM) 法
LM 算法²⁰ 是求解非线性最小二乘问题的黄金标准³⁹。

- **方法**：LM 算法通过一个阻尼因子 $\lambda$ ⁴⁰，巧妙地在“梯度下降法”（在远离最优解时稳定下降）⁴⁰ 和“高斯-牛顿法”（在接近最优解时快速收敛）²⁰ 之间插值。
- **优点**：非常鲁棒（robust）²⁰，即使初始猜测“非常偏离”最终最小值，也常常能找到解²⁰。
- **缺点**：
    - **局部最小值**：LM 只能保证收敛到**局部**最小值²⁰，这个解完全取决于你的初始猜测 $q_0$ ⁵。
    - **单一解**：每次运行只能得到一个解。


### C. 基于学习的求解器（Learning-Based Solvers）

#### 1. 核心动机
传统方法在处理高自由度（冗余）机器人时面临挑战⁴³。对于7-DOF以上的冗余臂，存在无限解集¹³。问题不再是“找到一个解”，而是“从无限解中找到最优解”¹³。这自然地导向了机器学习方法⁴³。

#### 2. ikflow：学习解的多样性
ikflow⁵⁰ 是解决此问题的一种先进的生成式建模方法⁴⁷。

- **方法**：它使用“条件归一化流”（Conditional Normalizing Flows, C-NF）⁴⁷。这是一种深度生成模型，它学习从一个简单的基础分布（如高斯分布）到复杂的目标分布（给定笛卡尔位姿下的整个IK解空间）的映射⁴⁸。
- **优点**：
    - **多样性与速度**：ikflow 可以在毫秒级别内生成**数千个**⁴⁹**多样化**的IK解⁵⁰。这完美解决了冗余臂的无限解采样问题¹²。
    - **通用性**：与数值法一样，只要有FK模型就可以训练，非常通用。
- **缺点**：
    - **精度低**：解的精度相对较低（例如，毫米级的位置误差和1.5度左右的姿态误差）⁴⁹。
    - **非Zero-shot**：必须为每个新机器人（或新配置）重新训练⁴⁷。

#### 3. 总结：学习与传统的共生
基于学习的算法⁵¹ 和传统算法⁵⁴ 不是非此即彼的竞争关系，而是**互补的**⁵¹。

- **ikflow** 擅长**全局采样**（快速生成大量、多样的低精度解）⁴⁹。
- **LM** 擅长**局部精化**（从一个好的初始猜测快速收敛到高精度解）²⁰。

因此，最高效的系统将两者结合：**使用ikflow生成多个初始猜测，然后并行运行LM将它们精炼为高精度解**。这种混合范式（Hybrid paradigm）同时解决了LM的“局部最小值”和“单一解”的核心弱点，以及ikflow的“低精度”弱点。


## IV. 应对现代操纵挑战的先进求解器

基于上述挑战，一系列先进的求解器被开发出来，它们分别代表了“几何”、“优化”和“混合”三种解决思路。


### A. 几何复兴：自动子问题分解

这类求解器试图将解析法的速度和完备性与数值法的通用性（在6R的类别内）结合起来。

#### EAIK (Efficient Analytical Inverse Kinematics)
EAIK⁶¹ 是这一理念的先进代表⁵⁹。

- **核心创新**：自动化的几何分解（Automatic Geometric Decomposition）⁵⁹。
- **方法**：EAIK 能自动“重塑”（remodel）运动链⁶¹。它通过“几何特征匹配”——即自动检测URDF中的平行和相交的关节轴——来自动将机器人分配到一个已知的“运动学类别”⁵⁹。这个类别对应于一个预先解决的子问题分解序列⁵⁹。
- **优点**：EAIK 首次实现了毫秒级的分析性IK推导和计算⁶⁰。它消除了ikfast所需的（可能很长的）“编译”步骤，也消除了传统解析法的手动推导。它在速度和精度上能匹配甚至超越ikfast⁶⁰。


### B. 先进的数值与优化求解器

#### 1. uwgraphics/relaxed_ik
- **问题**：如 II.B 所述，“最佳”IK是一个多目标问题。标准IK求解器只关注一个目标：位姿匹配。
- **方法**：RelaxedIK⁶⁶ 将IK问题明确地表述为一个加权和非线性优化（weighted-sum non-linear optimization）问题⁶⁶。优化的目标函数⁶⁶ 是多个“运动特性”的加权和，包括：末端位姿匹配、关节平滑度、远离自碰撞、远离奇异点等。
- **“Relaxed”的含义**：其核心思想是，末端位姿精度只是众多目标中的一个。如果它与其他更重要的目标（如避免碰撞）冲突，它的权重可以被“放松”（Relaxed）⁶⁶。这使得RelaxedIK非常适合生成平滑、可行的实时运动，而不是死板地追求亚毫米级的位姿精度。

<figure style="text-align: center; margin: 20px 0;">
    <video controls style="display: block; margin: 0 auto; max-width: 90%; height: auto;" autoplay muted loop>
        <source src="http://graphics.cs.cmu.edu/projects/relaxed-ik/videos/final-video-fast.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            RelaxedIK：机器人为了保持运动平滑和远离障碍物，可以“放松”对末端位姿的精确跟踪要求。
    </figcaption>
</figure>

#### 2. 层次二次规划（Hierarchical Quadratic Programming, HQP）
- **问题**：与RelaxedIK类似，HQP也用于解决多目标问题。但RelaxedIK使用“加权和”（软优先级），如果权重设置不当，次要任务仍可能干扰主要任务。
- **方法**：HQP⁷¹ 提供了一种**严格的优先级排序**，称为“任务栈”（Stack of Tasks, SoT）⁷¹。求解器首先只为最高优先级的任务（例如避免碰撞）找到最优解。然后，它在不影响最高优先级任务解的前提下，在第一任务的**零空间**（null-space）内求解第二优先级的任务（例如末端位姿）⁷¹。
- **对比 (HQP vs. RelaxedIK)**：这是两种解决多目标IK问题的不同哲学。RelaxedIK（加权和）更“平滑”，在所有目标之间进行“权衡”。HQP（零空间投影）更“严格”，能保证高优先级任务的实现，是安全关键（safety-critical）应用（如人机协作）的理想选择。


### C. 生成式与混合求解器（Generative and Hybrid Solvers）

#### jstmn/cppflow

CppFlow (ICRA 2024)⁷⁵ 是一个用于笛卡尔路径规划（CPP）的高性能规划器⁷⁶，它完美地体现了 III.C.3 中提到的**混合范式**。它是一个“**生成-选择-精化**”（Generate-Select-Refine）架构⁷⁶，结合了学习、规划和优化的优点。

其方法论如下⁷⁹：
1.  **生成（Generate）**：CppFlow 的核心是使用一个学习到的生成式IK求解器（即ikflow）²⁴。它在GPU上并行生成 `$K$` 个完整的候选轨迹⁷⁶。
2.  **选择（Select）**：CppFlow 接着使用一种经典方法——全局离散搜索⁷⁶ 或动态规划——来从这 `$K$` 条候选轨迹中，搜索出一条避免碰撞且运动最平滑的最优路径 $\xi$ ⁷⁹。
3.  **精化（Refine）**：最后，CppFlow 再次使用一种经典方法——Levenberg-Marquardt（LM）优化器⁷⁶——来获取这条被选中的、平滑但近似的路径 $\xi$，并将其精炼为满足高精度和约束的精确解⁷⁶。

CppFlow ⁷⁶ 展现了最前沿的思路：利用AI（ikflow）解决其擅长的全局采样问题，利用经典规划（Graph Search）解决其擅长的组合优化（序列选择）问题，利用经典数值（LM）解决其擅长的局部精化（高精度）问题。

<figure style="text-align: center; margin: 20px 0;">
    <video controls style="display: block; margin: 0 auto; max-width: 90%; height: auto;" autoplay muted loop>
        <source src="https://www.joeltmurphys.com/s/compare-side-4x-h264.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            CppFlow 与其他先进规划器在复杂路径上的性能对比。
    </figcaption>
</figure>


## V. 综合与未来展望

本章探讨了逆运动学（IK）从一个简单的几何求解问题，演变为TAMP框架下一个复杂的多目标、序列优化和可行性检查问题的过程。

### A. 比较分析：为TAMP选择正确的求解器

为TAMP的不同阶段选择合适的工具至关重要。下表总结了本章讨论的先进求解器架构及其在TAMP中的最佳应用场景。

| **求解器** | **核心方法论** | **主要任务** | **解输出** | **精度** | **在TAMP中的核心优势** |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **ikfast** | 自动化解析（C++代码生成）³¹ | 位姿求解 | 全解（针对6-DOF） | 高 | 极快的TAMP可行性检查³¹。 |
| **QuIK** | 高阶数值（三阶哈雷法）⁶⁴ | 位姿求解（实时） | 单解（局部） | 高 | 可靠的 1 kHz 实时跟踪与伺服⁶³。 |
| **ik-geo/EAIK** | 解析（子问题分解）⁵⁶ | 位姿求解 | 全解（针对6R）⁵⁶ | 高 | 通用、鲁棒且快速的解析解，无需手动推导或离线编译⁶⁰。 |
| **RelaxedIK** | 优化（加权和）⁶⁶ | 路径（实时） | 单解（优化） | 可调 | 同时求解位姿与约束（碰撞、平滑度），允许为可行性“放松”精度⁶⁶。 |
| **HQP** | 优化（零空间投影）⁷¹ | 路径（实时） | 单解（优化） | 高 | 对TAMP约束（如安全）提供严格的优先级保证⁷³。 |
| **ikflow** | 学习（生成式C-NF）⁴⁷ | 位姿求解 | 多样化采样（大量）⁵⁰ | 低 | 解决了7+DOF冗余臂的采样问题⁴⁹；为TAMP找到多样化的可行起点。 |
| **CppFlow** | 混合（ikflow + 搜索 + LM）⁷⁶ | 路径规划 | 单一最优路径 | 高 | 鲁棒地求解困难的笛卡尔路径问题⁷⁶；结合了学习的速度和经典方法的精度⁷⁶。 |

### B. 未来趋势：几何、优化与学习的融合

如近期的系统综述所强调⁵¹，IK的未来不属于任何单一方法，而在于**混合（Hybrid）方法**⁵¹。

- **AI作为加速器，而非替代品**：基于学习的方法⁴⁵正被证明是强大的采样器和启发式提供者。它们为经典规划器和优化器⁸⁵提供了更好的“起点”和“指导”，而不是完全取代它们。
- **标准化基准**：随着AI方法的普及，一个关键的挑战是建立标准化的基准⁵⁴，特别是在AI-IK领域，以确保结果的可复现性和在工业界（如机器人手术）⁵⁴ 的可靠应用。
- **新领域**：这些IK技术正在被扩展到更具挑战性的领域，如软体机器人（Soft Robotics）⁸⁰（其模型不再是刚体连杆，而是偏微分方程）和并联机器人（Parallel Robots）⁸⁰（其FK困难，但IK相对简单）。

总之，运动学仍然是机器人操纵的核心。然而，TAMP的出现已经将其从一个纯粹的几何问题，转变为一个集几何、优化、规划和学习于一体的复杂挑战。
